# E-Commerce Analytics — Exploratory Data Analysis (EDA)

An end-to-end analysis of a synthetic e-commerce order book covering **sales**, **profit**, **products**, **customers**, **geography** and **discounts**.

**Dataset**: synthetic `ecommerce_dataset.csv` (5,500 cleaned transactions, 2019–2023).
**Pipeline**: Raw Dataset → Data Cleaning → EDA → SQL Analysis → Tableau Dashboard → Business Insights.

This notebook focuses on the **EDA stage**. It mirrors the exact numbers used by the SQL queries and the Tableau dashboard so all deliverables stay consistent.

## 0. Setup & Configuration

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Locate the project root regardless of the current working directory
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_PATH = ROOT / "outputs" / "cleaned_data" / "ecommerce_clean.csv"
CHART_DIR = ROOT / "outputs" / "charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)

# Consistent, portfolio-ready chart style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.figsize": (10, 5.5),
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "figure.dpi": 120,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
})

def save_chart(fig, name):
    """Save a matplotlib figure to the charts folder."""
    path = CHART_DIR / name
    fig.savefig(path)
    plt.close(fig)
    print(f"Saved chart: {path.name}")

In [2]:
df = pd.read_csv(DATA_PATH, parse_dates=["Order Date"])
print(f"Cleaned dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

Cleaned dataset loaded: 5,500 rows x 25 columns


## 1. Data Overview

In [3]:
print("--- Data types ---")
print(df.dtypes.to_string())
print("\n--- Missing values ---")
print(df.isna().sum().to_string())
print("\n--- Duplicates ---")
print(f"Duplicate rows: {df.duplicated().sum()}")

--- Data types ---
Order ID                        str
Order Date           datetime64[us]
Customer ID                     str
Customer Name                   str
Segment                         str
Country                         str
State                           str
City                            str
Region                          str
Product ID                      str
Product Name                    str
Category                        str
Sub-Category                    str
Sales                       float64
Quantity                      int64
Discount                    float64
Profit                      float64
Shipping Cost               float64
Shipping Mode                   str
Payment Mode                    str
Year                          int64
Month                         int64
Quarter                       int64
Order Year-Month                str
Profit Margin (%)           float64

--- Missing values ---
Order ID             0
Order Date           0
Customer ID

In [4]:
pd.set_option("display.max_columns", None)
df.head(5)

,Order ID,Order Date,Customer ID,Customer Name,Segment,Country,State,City,Region,Product ID,Product Name,Category,Sub-Category,Sales,Quantity,Discount,Profit,Shipping Cost,Shipping Mode,Payment Mode,Year,Month,Quarter,Order Year-Month,Profit Margin (%)
0,CA-2019-14748,2019-10-24,CUS-0632,Carol Ramos,Consumer,United States,Virginia,Arlington,East,FUR-0032,Summit Chairs 951,Furniture,Chairs,992.70,2,0.00,185.91,38.04,Second Class,Cash on Delivery,2019,10,4,2019-10,18.73
1,CA-2019-11200,2019-02-25,CUS-0436,Jessica Bennett,Consumer,United States,Massachusetts,Boston,East,OFF-0099,Ergo Art 414,Office Supplies,Art,62.97,1,0.30,1.65,9.53,Second Class,UPI,2019,2,1,2019-02,2.62
2,CA-2020-15223,2020-07-23,CUS-0483,Paul Brooks,Consumer,United States,Minnesota,Minneapolis,Central,OFF-0064,Vertex Binders 563,Office Supplies,Binders,141.08,2,0.15,26.11,13.38,First Class,UPI,2020,7,3,2020-07,18.51
3,CA-2022-14548,2022-09-25,CUS-0101,Ella Reddy,Consumer,United States,New Jersey,Newark,East,OFF-0073,Vertex Paper 524,Office Supplies,Paper,100.23,2,0.10,17.39,10.45,Second Class,Debit Card,2022,9,3,2022-09,17.35
4,CA-2021-11863,2021-12-25,CUS-0453,Gary Sanders,Corporate,United States,Missouri,Kansas City,Central,OFF-0118,Ergo Supplies 345,Office Supplies,Supplies,57.29,2,0.20,11.29,5.41,Standard Class,UPI,2021,12,4,2021-12,19.71


In [5]:
df.describe().round(2)

,Order Date,Sales,Quantity,Discount,Profit,Shipping Cost,Year,Month,Quarter,Profit Margin (%)
count,5500,5500.00,5500.00,5500.00,5500.00,5500.00,5500.00,5500.00,5500.00,5500.00
mean,2021-07-08 22:48:47.127272,783.70,3.28,0.17,107.11,30.03,2020.99,6.93,2.65,12.43
min,2019-01-01 00:00:00,8.39,1.00,0.00,-468.50,5.08,2019.00,1.00,1.00,-28.52
25%,2020-04-03 18:00:00,112.55,2.00,0.00,7.59,12.31,2020.00,4.00,2.00,6.77
50%,2021-07-18 12:00:00,325.36,3.00,0.15,29.23,20.89,2021.00,7.00,3.00,13.72
75%,2022-10-12 00:00:00,1036.34,4.00,0.30,111.02,34.44,2022.00,10.00,4.00,19.49
max,2023-12-28 00:00:00,8880.67,7.00,0.50,2826.43,418.76,2023.00,12.00,4.00,39.25
std,NaN,1081.06,1.76,0.17,221.59,31.47,1.43,3.44,1.11,10.06


## 2. Sales Analysis

In [6]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_orders = df["Order ID"].nunique()
total_quantity = df["Quantity"].sum()
aov = total_sales / total_orders
profit_margin = total_profit / total_sales * 100

kpis = pd.DataFrame({
    "KPI": ["Total Sales", "Total Profit", "Total Orders", "Total Quantity",
            "Average Order Value (AOV)", "Profit Margin"],
    "Value": [f"${total_sales:,.0f}", f"${total_profit:,.0f}", f"{total_orders:,}",
               f"{total_quantity:,}", f"${aov:,.2f}", f"{profit_margin:.1f}%"],
})
kpis

,KPI,Value
0,Total Sales,"$4,310,325"
1,Total Profit,"$589,096"
2,Total Orders,"5,500"
3,Total Quantity,"18,051"
4,Average Order Value (AOV),$783.70
5,Profit Margin,13.7%


In [7]:
monthly = df.groupby(df["Order Date"].dt.to_period("M")).agg(
    Sales=("Sales", "sum"), Profit=("Profit", "sum")
).reset_index()
monthly["Year-Month"] = monthly["Order Date"].astype(str)

fig, ax1 = plt.subplots()
ax1.plot(monthly["Year-Month"], monthly["Sales"] / 1_000, color="#1f77b4", linewidth=2,
         marker="o", markersize=3, label="Sales ($K)")
ax1.set_xlabel("Year-Month")
ax1.set_ylabel("Sales ($K)", color="#1f77b4")
ax1.tick_params(axis="y", labelcolor="#1f77b4")
ax1.tick_params(axis="x", rotation=90)

ax2 = ax1.twinx()
ax2.plot(monthly["Year-Month"], monthly["Profit"] / 1_000, color="#d62728", linewidth=1.6,
         marker="s", markersize=3, label="Profit ($K)")
ax2.set_ylabel("Profit ($K)", color="#d62728")
ax2.tick_params(axis="y", labelcolor="#d62728")

ax1.set_title("Monthly Sales & Profit Trend (2019-2023)")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
save_chart(fig, "monthly_sales_profit_trend.png")
plt.show()

print("Top 3 months by sales:")
print(monthly.sort_values("Sales", ascending=False).head(3).to_string(index=False))

Saved chart: monthly_sales_profit_trend.png
Top 3 months by sales:
Order Date     Sales   Profit Year-Month
   2020-11 113003.81 17411.13    2020-11
   2023-11 110280.73 15760.60    2023-11
   2020-12 100968.71 14617.74    2020-12


In [8]:
yearly = df.groupby(df["Order Date"].dt.year).agg(
    Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order ID", "nunique")
).reset_index().rename(columns={"Order Date": "Year"})

fig, ax = plt.subplots()
x = np.arange(len(yearly))
width = 0.38
ax.bar(x - width / 2, yearly["Sales"] / 1_000, width, label="Sales ($K)", color="#1f77b4")
ax.bar(x + width / 2, yearly["Profit"] / 1_000, width, label="Profit ($K)", color="#2ca02c")
ax.set_xticks(x, yearly["Year"])
ax.set_xlabel("Year")
ax.set_ylabel("Amount ($K)")
ax.set_title("Yearly Sales & Profit")
ax.legend()
for i in range(len(yearly)):
    ax.text(x[i] - width / 2, yearly["Sales"][i] / 1_000 + 20, f"{yearly['Sales'][i]/1_000:.0f}K",
            ha="center", fontsize=8)
    ax.text(x[i] + width / 2, yearly["Profit"][i] / 1_000 + 20, f"{yearly['Profit'][i]/1_000:.0f}K",
            ha="center", fontsize=8)
save_chart(fig, "yearly_sales_profit.png")
plt.show()

yearly

Saved chart: yearly_sales_profit.png


,Year,Sales,Profit,Orders
0,2019,963313.28,133363.06,1145
1,2020,904117.84,129508.59,1089
2,2021,787370.81,102299.02,1070
3,2022,771021.83,105096.46,1072
4,2023,884500.98,118828.52,1124


## 3. Product Analysis

In [9]:
category = df.groupby("Category").agg(
    Sales=("Sales", "sum"), Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique"), Quantity=("Quantity", "sum")
).round(2).sort_values("Sales", ascending=False)
category["Profit Margin (%)"] = (category["Profit"] / category["Sales"] * 100).round(1)
category

,Sales,Profit,Orders,Quantity,Profit Margin (%)
Category,,,,,
Technology,2190724.06,356101.86,1364,4642,16.3
Furniture,1306820.61,104555.27,1387,4567,8.0
Office Supplies,812780.07,128438.52,2749,8842,15.8


In [10]:
fig, ax = plt.subplots()
x = np.arange(len(category))
width = 0.38
ax.bar(x - width / 2, category["Sales"] / 1_000, width, label="Sales ($K)", color="#1f77b4")
ax.bar(x + width / 2, category["Profit"] / 1_000, width, label="Profit ($K)", color="#2ca02c")
ax.set_xticks(x, category.index)
ax.set_xlabel("Category")
ax.set_ylabel("Amount ($K)")
ax.set_title("Category-wise Sales vs Profit")
ax.legend()
save_chart(fig, "category_sales_profit.png")
plt.show()

Saved chart: category_sales_profit.png


In [11]:
top_products_sales = (df.groupby("Product Name")
                      .agg(Sales=("Sales", "sum"))
                      .sort_values("Sales", ascending=False).head(10))

fig, ax = plt.subplots()
top_products_sales.sort_values("Sales").plot.barh(
    ax=ax, color="#1f77b4", legend=False, width=0.7)
ax.set_xlabel("Total Sales ($)")
ax.set_title("Top 10 Products by Sales")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v/1_000:.0f}K"))
save_chart(fig, "top10_products_sales.png")
plt.show()
top_products_sales.round(2)

Saved chart: top10_products_sales.png


,Sales
Product Name,
Nova Copiers 963,189026.89
Vertex Copiers 222,184871.88
Zenith Copiers 734,145345.08
TechPro Copiers 202,143662.68
Nova Copiers 555,139422.91
Apex Copiers 621,126733.01
Nova Tables 283,121562.15
Zenith Tables 379,110535.94
Summit Copiers 502,108877.61


In [12]:
top_products_profit = (df.groupby("Product Name")
                       .agg(Profit=("Profit", "sum"))
                       .sort_values("Profit", ascending=False).head(10))

fig, ax = plt.subplots()
top_products_profit.sort_values("Profit").plot.barh(
    ax=ax, color="#2ca02c", legend=False, width=0.7)
ax.set_xlabel("Total Profit ($)")
ax.set_title("Top 10 Products by Profit")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v/1_000:.0f}K"))
save_chart(fig, "top10_products_profit.png")
plt.show()
top_products_profit.round(2)

Saved chart: top10_products_profit.png


,Profit
Product Name,
Nova Copiers 963,30184.60
Vertex Copiers 222,27716.02
Nova Copiers 555,25951.47
Zenith Copiers 734,24557.94
Apex Copiers 621,21544.41
TechPro Copiers 202,21272.16
Nova Machines 941,17453.46
Summit Copiers 502,16581.79
Prime Machines 988,16471.97


In [13]:
bottom_products_profit = (df.groupby("Product Name")
                          .agg(Profit=("Profit", "sum"))
                          .sort_values("Profit").head(10))

fig, ax = plt.subplots()
bottom_products_profit.sort_values("Profit").plot.barh(
    ax=ax, color="#d62728", legend=False, width=0.7)
ax.set_xlabel("Total Profit ($)")
ax.set_title("Bottom 10 Products by Profit (Lowest Profit)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v/1_000:.0f}K"))
save_chart(fig, "bottom10_products_profit.png")
plt.show()
bottom_products_profit.round(2)

Saved chart: bottom10_products_profit.png


,Profit
Product Name,
Apex Fasteners 348,268.93
Prime Labels 527,325.66
TechPro Fasteners 845,367.53
Ergo Envelopes 904,387.97
Orbit Labels 667,392.66
Nova Supplies 494,430.05
Luma Fasteners 480,433.34
Nova Labels 521,446.92
Orbit Envelopes 238,467.66


In [14]:
most_sold = (df.groupby("Product Name")
             .agg(Quantity=("Quantity", "sum"), Sales=("Sales", "sum"))
             .sort_values("Quantity", ascending=False).head(10))
print("Top 10 most-sold products (by units sold):")
most_sold.round(2)

Top 10 most-sold products (by units sold):


,Quantity,Sales
Product Name,,
Summit Furnishings 293,246,18164.60
Nova Copiers 963,239,189026.89
Vertex Copiers 222,214,184871.88
Zenith Accessories 482,213,18892.65
Pioneer Appliances 560,210,68634.90
Nova Accessories 966,207,21225.00
Prime Machines 988,204,100182.77
Luma Fasteners 285,198,3650.81
Nova Tables 283,197,121562.15


In [15]:
subcat = (df.groupby(["Category", "Sub-Category"])
          .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
          .sort_values("Sales", ascending=False).head(10))

fig, ax = plt.subplots()
idx = [f"{c} | {s}" for c, s in subcat.index]
subcat_plot = subcat.copy()
subcat_plot.index = idx
subcat_plot.sort_values("Sales").plot.barh(ax=ax, color=["#1f77b4", "#2ca02c"], width=0.7)
ax.set_xlabel("Amount ($)")
ax.set_title("Top 10 Sub-Categories by Sales (Sales vs Profit)")
ax.legend(loc="lower right")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v/1_000:.0f}K"))
save_chart(fig, "subcategory_performance.png")
plt.show()
subcat.round(2)

Saved chart: subcategory_performance.png


Sales     Profit
Category        Sub-Category                       
Technology      Copiers       1236883.18  197874.73
Furniture       Tables         588989.26   49189.96
Technology      Machines       448218.28   78246.27
Office Supplies Appliances     445153.91   70195.70
Technology      Phones         375938.73   58681.02
Furniture       Chairs         332279.08   25751.12
                Bookcases      297542.23   22106.63
Technology      Accessories    129683.87   21299.84
Office Supplies Storage         95597.74   15007.85
                Art             92269.48   14821.70

## 4. Customer Analysis

In [16]:
n_customers = df["Customer ID"].nunique()
print(f"Number of unique customers: {n_customers:,}")
print(f"Average order value per customer: ${total_sales / n_customers:,.2f}")

top_customers = (df.groupby(["Customer ID", "Customer Name"])
                 .agg(Sales=("Sales", "sum"), Orders=("Order ID", "nunique"))
                 .reset_index()
                 .sort_values("Sales", ascending=False).head(10))

fig, ax = plt.subplots()
labels = [f"{n[:10]}..." if len(n) > 10 else n for n in top_customers["Customer Name"][::-1]]
ax.barh(labels, top_customers["Sales"][::-1] / 1_000, color="#9467bd")
ax.set_xlabel("Total Sales ($K)")
ax.set_title("Top 10 Customers by Sales")
save_chart(fig, "top10_customers_sales.png")
plt.show()
top_customers.round(2)

Number of unique customers: 649
Average order value per customer: $6,641.49


Saved chart: top10_customers_sales.png


,Customer ID,Customer Name,Sales,Orders
316,CUS-0317,Isabella Morris,24275.76,14
480,CUS-0482,Ella Morgan,22979.69,11
470,CUS-0472,Tyler Phillips,20324.79,16
67,CUS-0068,Melissa Gomez,18584.80,9
73,CUS-0074,Jessica Gonzalez,17906.54,12
530,CUS-0532,Daniel Patel,16535.01,9
160,CUS-0161,Kavya Jones,16476.20,13
590,CUS-0592,Arjun Richardson,16012.30,12
309,CUS-0310,Susan Nelson,15935.54,15
164,CUS-0165,Timothy Richardson,15861.05,14


In [17]:
(df.groupby(["Customer ID", "Customer Name"])
 .agg(Profit=("Profit", "sum"))
 .sort_values("Profit", ascending=False).head(10).round(2))

,,Profit
Customer ID,Customer Name,
CUS-0074,Jessica Gonzalez,3909.62
CUS-0409,Ananya Baker,3862.93
CUS-0334,Ella Gutierrez,3700.19
CUS-0317,Isabella Morris,3579.49
CUS-0138,Patricia Cooper,3562.14
CUS-0181,Ravi Sanchez,3474.69
CUS-0472,Tyler Phillips,3401.23
CUS-0482,Ella Morgan,3276.56
CUS-0125,Adam Gray,3088.41


In [18]:
segments = df.groupby("Segment").agg(
    Sales=("Sales", "sum"), Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique"), Customers=("Customer ID", "nunique")
).round(2).sort_values("Sales", ascending=False)
segments["Profit Margin (%)"] = (segments["Profit"] / segments["Sales"] * 100).round(1)

fig, ax = plt.subplots()
x = np.arange(len(segments))
width = 0.38
ax.bar(x - width / 2, segments["Sales"] / 1_000, width, label="Sales ($K)", color="#ff7f0e")
ax.bar(x + width / 2, segments["Profit"] / 1_000, width, label="Profit ($K)", color="#2ca02c")
ax.set_xticks(x, segments.index)
ax.set_ylabel("Amount ($K)")
ax.set_title("Customer Segment Performance (Sales vs Profit)")
ax.legend()
save_chart(fig, "segment_sales_profit.png")
plt.show()
segments

Saved chart: segment_sales_profit.png


,Sales,Profit,Orders,Customers,Profit Margin (%)
Segment,,,,,
Consumer,2222760.50,300997.44,2840,340,13.5
Corporate,1234458.76,173770.01,1596,187,14.1
Home Office,853105.48,114328.20,1064,122,13.4


## 5. Geographic Analysis

In [19]:
region = df.groupby("Region").agg(
    Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order ID", "nunique")
).round(2).sort_values("Sales", ascending=False)
region["Profit Margin (%)"] = (region["Profit"] / region["Sales"] * 100).round(1)

fig, ax = plt.subplots()
x = np.arange(len(region))
width = 0.38
ax.bar(x - width / 2, region["Sales"] / 1_000, width, label="Sales ($K)", color="#17becf")
ax.bar(x + width / 2, region["Profit"] / 1_000, width, label="Profit ($K)", color="#2ca02c")
ax.set_xticks(x, region.index)
ax.set_ylabel("Amount ($K)")
ax.set_title("Region-wise Sales vs Profit")
ax.legend()
save_chart(fig, "region_sales_profit.png")
plt.show()
region

Saved chart: region_sales_profit.png


,Sales,Profit,Orders,Profit Margin (%)
Region,,,,
Central,1128369.41,152663.45,1426,13.5
East,1094392.15,148201.85,1354,13.5
West,1082161.08,155823.33,1331,14.4
South,1005402.10,132407.02,1389,13.2


In [20]:
print(f"Country distribution (transactions): {df['Country'].value_counts().to_dict()}")

state = df.groupby("State").agg(Sales=("Sales", "sum")).sort_values("Sales", ascending=False).head(10)
fig, ax = plt.subplots()
state.sort_values("Sales").plot.barh(ax=ax, color="#8c564b", legend=False, width=0.7)
ax.set_xlabel("Total Sales ($)")
ax.set_title("Top 10 States by Sales")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v/1_000:.0f}K"))
save_chart(fig, "state_sales.png")
plt.show()
state.round(2)

Country distribution (transactions): {'United States': 5500}


Saved chart: state_sales.png


,Sales
State,
New Jersey,186909.25
Illinois,186548.50
Maryland,177029.95
Virginia,173815.11
Washington,172817.69
Texas,171844.76
Florida,170752.88
Utah,170752.73
Wisconsin,169896.38


In [21]:
city = df.groupby("City").agg(Sales=("Sales", "sum")).sort_values("Sales", ascending=False).head(10)
fig, ax = plt.subplots()
city.sort_values("Sales").plot.barh(ax=ax, color="#7f7f7f", legend=False, width=0.7)
ax.set_xlabel("Total Sales ($)")
ax.set_title("Top 10 Cities by Sales")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v/1_000:.0f}K"))
save_chart(fig, "city_sales.png")
plt.show()
city.round(2)

Saved chart: city_sales.png


,Sales
City,
Springfield,138288.56
Rochester,76866.93
Salem,73945.56
Provo,72106.38
Green Bay,69936.26
Madison,68779.71
Silver Spring,67343.09
Baltimore,67059.58
Overland Park,64791.91


## 6. Discount Analysis

In [22]:
fig, ax = plt.subplots()
sns.histplot(df["Discount"], bins=20, kde=False, color="#ff7f0e", ax=ax)
ax.set_xlabel("Discount")
ax.set_ylabel("Number of Orders")
ax.set_title("Distribution of Discounts")
save_chart(fig, "discount_distribution.png")
plt.show()

print("Discount share of orders:")
print(df["Discount"].value_counts(normalize=True).sort_index().round(4).to_string())

Saved chart: discount_distribution.png
Discount share of orders:
Discount
0.00    0.3738
0.10    0.0962
0.15    0.0880
0.20    0.0905
0.25    0.0840
0.30    0.0889
0.40    0.0891
0.50    0.0895


In [23]:
df["Discount Bucket"] = pd.cut(df["Discount"], [-0.01, 0.01, 0.2, 0.4, 0.6],
                                labels=["0%", "Up to 20%", "20-40%", "40%+"])

disc_by_bucket = df.groupby("Discount Bucket", observed=True).agg(
    Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order ID", "nunique")
).round(2)
disc_by_bucket["Profit Margin (%)"] = (disc_by_bucket["Profit"] / disc_by_bucket["Sales"] * 100).round(1)

fig, ax = plt.subplots()
x = np.arange(len(disc_by_bucket))
width = 0.38
ax.bar(x - width / 2, disc_by_bucket["Sales"] / 1_000, width, label="Sales ($K)", color="#1f77b4")
ax.bar(x + width / 2, disc_by_bucket["Profit"] / 1_000, width, label="Profit ($K)", color="#d62728")
ax.set_xticks(x, disc_by_bucket.index)
ax.set_xlabel("Discount Level")
ax.set_ylabel("Amount ($K)")
ax.set_title("Discount Level vs Sales & Profit")
ax.legend()
save_chart(fig, "discount_vs_profit.png")
plt.show()
disc_by_bucket

Saved chart: discount_vs_profit.png


,Sales,Profit,Orders,Profit Margin (%)
Discount Bucket,,,,
0%,1954970.55,400373.35,2056,20.5
Up to 20%,1220060.26,159791.26,1511,13.1
20-40%,911365.92,42211.44,1441,4.6
40%+,223928.01,-13280.40,492,-5.9


In [24]:
corr = df[["Discount", "Profit Margin (%)", "Sales", "Profit"]].corr()
print("Correlation between Discount and other metrics:")
print(corr["Discount"].round(3).to_string())

loss_rate_by_discount = df.assign(
    Loss=df["Profit"] < 0
).groupby("Discount Bucket", observed=True)["Loss"].mean().round(3) * 100
print("\nShare of loss-making orders by discount level (%):")
print(loss_rate_by_discount.astype(str).to_string())

Correlation between Discount and other metrics:
Discount             1.000
Profit Margin (%)   -0.736
Sales               -0.151
Profit              -0.354

Share of loss-making orders by discount level (%):
Discount Bucket
0%                          0.0
Up to 20%                   2.6
20-40%                     21.7
40%+         53.300000000000004


In [25]:
print("Products with the highest average discount:")
(df.groupby(["Product ID", "Product Name", "Category"])
 .agg(Avg_Discount=("Discount", "mean"), Sales=("Sales", "sum"), Profit=("Profit", "sum"))
 .sort_values("Avg_Discount", ascending=False).head(10).round(3))

Products with the highest average discount:


,,,Avg_Discount,Sales,Profit
Product ID,Product Name,Category,,,
FUR-0031,Apex Chairs 409,Furniture,0.232,55773.47,2336.06
OFF-0063,Prime Binders 977,Office Supplies,0.232,5954.96,924.27
OFF-0061,Zenith Binders 650,Office Supplies,0.219,7870.33,1053.42
OFF-0118,Ergo Supplies 345,Office Supplies,0.218,3012.09,467.67
OFF-0098,Luma Art 578,Office Supplies,0.214,9267.22,1149.60
TEC-0016,Pioneer Accessories 889,Technology,0.211,13657.24,1900.77
FUR-0043,TechPro Tables 104,Furniture,0.209,60995.47,3105.14
OFF-0065,Zenith Binders 978,Office Supplies,0.206,7597.29,1133.23
TEC-0022,Orbit Copiers 298,Technology,0.205,103759.97,14942.15


In [26]:
print("Average discount by category:")
df.groupby("Category")["Discount"].mean().round(3).to_string()

Average discount by category:


'Category\nFurniture          0.168\nOffice Supplies    0.169\nTechnology         0.170'

## 7. Seasonal Pattern (Heatmap)

In [27]:
pivot = df.pivot_table(index=df["Order Date"].dt.year, columns=df["Order Date"].dt.month,
                       values="Sales", aggfunc="sum") / 1_000
fig, ax = plt.subplots(figsize=(11, 4.5))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlGnBu", cbar_kws={"label": "Sales ($K)"}, ax=ax)
ax.set_xlabel("Month")
ax.set_ylabel("Year")
ax.set_title("Monthly Sales Heatmap ($K) — shows the year-end seasonality")
save_chart(fig, "monthly_sales_heatmap.png")
plt.show()

Saved chart: monthly_sales_heatmap.png


## 8. Business Questions — Answers

In [28]:
best_cat_sales = category["Sales"].idxmax()
best_cat_profit = category["Profit"].idxmax()
top_product_sales = top_products_sales.index[0]
worst_product_profit = (df.groupby("Product Name")["Profit"].sum()
                        .sort_values().index[0])
top_customer = (df.groupby("Customer Name")["Sales"].sum().sort_values(ascending=False).index[0])
best_region = region["Sales"].idxmax()
month_sales = df.groupby(df["Order Date"].dt.month)["Sales"].sum()
month_profit = df.groupby(df["Order Date"].dt.month)["Profit"].sum()
best_month_sales = int(month_sales.idxmax())
best_month_profit = int(month_profit.idxmax())

# Sub-category needing improvement = lowest profit margin among sub-categories
subcat_margin = (df.groupby(["Category", "Sub-Category"])
                 .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum")))
subcat_margin["Margin"] = subcat_margin["Profit"] / subcat_margin["Sales"] * 100
weakest_subcat = subcat_margin.sort_values("Margin").index[0]
best_segment = segments["Sales"].idxmax()

# Share of loss-making orders by discount level (reused in Q11)
loss_by_bucket = (df.assign(Loss=df["Profit"] < 0)
                  .groupby("Discount Bucket", observed=True)["Loss"].mean())

answers = pd.DataFrame({
    "Q": range(1, 15),
    "Business Question": [
        "What is the total revenue?",
        "What is the total profit?",
        "Which category generates the most revenue?",
        "Which category generates the most profit?",
        "Which products have the highest sales?",
        "Which products have the lowest profit?",
        "Which customers generate the most revenue?",
        "Which region performs best?",
        "Which month has the highest sales?",
        "Which month has the highest profit?",
        "Does higher discount reduce profit?",
        "Which sub-category needs improvement?",
        "What is the average order value?",
        "Which customer segment performs best?",
    ],
    "Answer": [
        f"${total_sales:,.0f}",
        f"${total_profit:,.0f}",
        best_cat_sales,
        best_cat_profit,
        top_product_sales,
        worst_product_profit,
        top_customer,
        f"{best_region} (${region['Sales'].max():,.0f})",
        f"Month {best_month_sales} (${month_sales.max():,.0f})",
        f"Month {best_month_profit} (${month_profit.max():,.0f})",
        (f"Yes - margin correlation {corr['Discount']['Profit Margin (%)']:.2f}. "
         f"Loss-making orders rise from {loss_by_bucket.iloc[0]*100:.0f}% ({loss_by_bucket.index[0]}) "
         f"to {loss_by_bucket.iloc[-1]*100:.0f}% ({loss_by_bucket.index[-1]})."),
        f"{weakest_subcat[0]} | {weakest_subcat[1]}",
        f"${aov:,.2f}",
        best_segment,
    ],
})
pd.set_option("display.max_colwidth", 90)
answers

,Q,Business Question,Answer
0,1,What is the total revenue?,"$4,310,325"
1,2,What is the total profit?,"$589,096"
2,3,Which category generates the most revenue?,Technology
3,4,Which category generates the most profit?,Technology
4,5,Which products have the highest sales?,Nova Copiers 963
5,6,Which products have the lowest profit?,Apex Fasteners 348
6,7,Which customers generate the most revenue?,Isabella Morris
7,8,Which region performs best?,"Central ($1,128,369)"
8,9,Which month has the highest sales?,"Month 11 ($474,321)"
9,10,Which month has the highest profit?,"Month 11 ($66,113)"


## 9. Key Takeaways & Next Steps

* **Sales** are concentrated in **Technology** (largest revenue) while **Office Supplies** delivers the healthiest margins.
* **Consumer** is the biggest segment; **Corporate** offers the highest-value orders.
* Sales peak in **November–December** — a clear year-end holiday seasonality.
* **High discounts destroy margin**: orders at 40%+ discount are mostly loss-making.
* The **West** region leads in sales; several Furniture sub-categories need margin review.

**Next steps** in the workflow: SQL analysis (`sql/ecommerce_analysis.sql`), the Tableau dashboard (`dashboard/Ecommerce_Dashboard.twb`) and the final business-insights report in the README.